In [ ]:
import os
import gc
import random
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from tqdm import tqdm

from model_calandra import SpatioTemporalDynamicClassifier


def setup_logging(log_dir="logs_calandra"):
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"train_calandra_{timestamp}.log")

    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(log_file, encoding="utf-8")
        ]
    )
    logger = logging.getLogger(__name__)
    logger.info(f"日志保存至: {log_file}")
    return logger


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


_NORMALIZE = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
_TO_TENSOR_NORM = T.Compose([T.ToTensor(), _NORMALIZE])
_TEST_TRANSFORM = T.Compose([
    T.Resize(256, interpolation=InterpolationMode.BILINEAR),
    T.CenterCrop(224),
    T.ToTensor(),
    _NORMALIZE,
])


class CalandraDataset(Dataset):

    def __init__(self, root_dir, mode="train"):
        self.root_dir = Path(root_dir)
        self.mode = mode
        self.samples = []

        subset_path = self.root_dir / mode
        gelA_dir = subset_path / "gelsightA"

        if not gelA_dir.exists():
            raise FileNotFoundError(f"找不到目录: {gelA_dir}")

        for gelA_path in sorted(gelA_dir.glob("*.png")):
            stem = gelA_path.stem

            if mode == "test" and "_during_" not in stem:
                continue

            label = 1 if "success" in stem else 0

            try:
                parts = stem.split("_")
                gel_idx = parts.index("gelsightA")
            except ValueError:
                continue

            gelB_name = "_".join(parts[:gel_idx] + ["gelsightB"] + parts[gel_idx+1:]) + ".png"
            rgb_name = "_".join(parts[:gel_idx] + ["kinectA", "rgb"] + parts[gel_idx+1:]) + ".png"

            gelB_path = subset_path / "gelsightB" / gelB_name
            rgb_path = subset_path / "kinectA_rgb" / rgb_name

            if gelB_path.exists() and rgb_path.exists():
                self.samples.append((str(rgb_path), str(gelA_path), str(gelB_path), label))

        print(f"[Calandra-{mode}] 有效样本: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rgb_path, gelA_path, gelB_path, label = self.samples[idx]

        rgb = Image.open(rgb_path).convert("RGB")
        gelA = Image.open(gelA_path).convert("RGB")
        gelB = Image.open(gelB_path).convert("RGB")

        if self.mode == "train":
            i, j, h, w = T.RandomResizedCrop.get_params(
                rgb, scale=(0.8, 1.0), ratio=(3/4, 4/3)
            )
            do_flip = random.random() < 0.5

            rgb = TF.resized_crop(rgb, i, j, h, w, (224, 224), interpolation=InterpolationMode.BILINEAR)
            gelA = TF.resized_crop(gelA, i, j, h, w, (224, 224), interpolation=InterpolationMode.BILINEAR)
            gelB = TF.resized_crop(gelB, i, j, h, w, (224, 224), interpolation=InterpolationMode.BILINEAR)

            if do_flip:
                rgb = TF.hflip(rgb)
                gelA = TF.hflip(gelA)
                gelB = TF.hflip(gelB)

            rgb = T.ColorJitter(0.2, 0.2, 0.2, 0.05)(rgb)

            rgb = _TO_TENSOR_NORM(rgb)
            gelA = _TO_TENSOR_NORM(gelA)
            gelB = _TO_TENSOR_NORM(gelB)

        else:
            rgb = _TEST_TRANSFORM(rgb)
            gelA = _TEST_TRANSFORM(gelA)
            gelB = _TEST_TRANSFORM(gelB)

        tac = torch.cat([gelA, gelB], dim=0)

        rgb = rgb.unsqueeze(0)
        tac = tac.unsqueeze(0)

        return rgb, tac, label


def build_optimizer(model, lr_backbone, lr_head, weight_decay):
    backbone_decay, backbone_no_decay = [], []
    head_decay, head_no_decay = [], []

    def is_backbone(name):
        return (
            name.startswith("vision_encoder.")
            or name.startswith("tactile_encoder.")
            or name.startswith("v_spatial_proj.")
            or name.startswith("t_proj.")
            or name.startswith("v3_spatial_proj.")
            or name.startswith("tac_fusion.")
        )

    def is_no_decay(name, p):
        if name.endswith(".bias") or p.ndim <= 1:
            return True
        if "norm" in name.lower() or "pos" in name.lower():
            return True
        return False

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if is_backbone(name):
            (backbone_no_decay if is_no_decay(name, p) else backbone_decay).append(p)
        else:
            (head_no_decay if is_no_decay(name, p) else head_decay).append(p)

    param_groups = [
        {"params": backbone_decay,    "lr": lr_backbone, "weight_decay": weight_decay},
        {"params": backbone_no_decay, "lr": lr_backbone, "weight_decay": 0.0},
        {"params": head_decay,        "lr": lr_head,     "weight_decay": weight_decay},
        {"params": head_no_decay,     "lr": lr_head,     "weight_decay": 0.0},
    ]

    return torch.optim.AdamW(
        [g for g in param_groups if len(g["params"]) > 0],
        betas=(0.9, 0.98),
        eps=1e-6
    )


def load_mambavision_pretrained(model, ckpt_path, verbose=True):
    if not os.path.exists(ckpt_path):
        print(f"找不到预训练权重: {ckpt_path}，使用随机初始化")
        return []

    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt.get("model", ckpt))
    model_state = model.state_dict()
    loaded = []

    for k, v in state.items():
        if "head" in k or "classifier" in k:
            continue

        for prefix in ["vision_encoder", "tactile_encoder"]:
            target_key = f"{prefix}.model.{k}"
            if target_key in model_state and model_state[target_key].shape == v.shape:
                model_state[target_key].copy_(v)
                loaded.append(target_key)

    model.load_state_dict(model_state)

    if verbose:
        print(f"[Pretrain] 成功加载 {len(loaded)} 个参数（视觉+触觉编码器均完整加载）")
    return loaded


@torch.no_grad()
def evaluate(model, loader, device, criterion, use_amp=True):
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0

    for rgb, tac, label in loader:
        rgb = rgb.to(device, non_blocking=True)
        tac = tac.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
            logits = model(rgb, tac)
            loss = criterion(logits, label)

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * label.size(0)
        total_correct += (preds == label).sum().item()
        total_count += label.size(0)

    avg_loss = total_loss / max(1, total_count)
    avg_acc = 100.0 * total_correct / max(1, total_count)
    return avg_loss, avg_acc


def main():
    logger = setup_logging()

    config = {
        "data_folder":     "",
        "num_classes":     2,
        "epochs":          50,
        "batch_size":      32,
        "num_workers":     8,
        "grad_clip":       1.0,
        "lr_backbone":     3e-5,
        "lr_head":         3e-4,
        "weight_decay":    1e-2,
        "label_smoothing": 0.1,
        "dropout":         0.3,
        "d_model":         256,
        "d_state":         16,
        "hierarchical":    False,
        "no_tgsa":         True,
        "pretrained_path": "mambavision_tiny_1k.pth.tar",
        "save_dir":        "checkpoints_calandra",
    }

    os.makedirs(config["save_dir"], exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    logger.info("=" * 60)
    logger.info("Feel of Success (Calandra) 训练")
    logger.info("触觉处理：gelA / gelB 分别编码后特征融合")
    logger.info(f"设备: {device}")
    for k, v in config.items():
        logger.info(f"  {k}: {v}")
    logger.info("=" * 60)

    logger.info("加载数据集...")
    train_dataset = CalandraDataset(config["data_folder"], mode="train")
    test_dataset = CalandraDataset(config["data_folder"], mode="test")

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"],
        drop_last=True,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=True,
    )

    logger.info(f"训练集: {len(train_dataset)} | 测试集: {len(test_dataset)}")

    logger.info("构建模型...")
    model = SpatioTemporalDynamicClassifier(
        num_classes=config["num_classes"],
        d_model=config["d_model"],
        d_state=config["d_state"],
        dropout=config["dropout"],
        hierarchical=config["hierarchical"],
        no_tgsa=config["no_tgsa"],
    ).to(device)

    if config["pretrained_path"] and os.path.exists(config["pretrained_path"]):
        load_mambavision_pretrained(model, config["pretrained_path"], verbose=True)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"可训练参数量: {total_params / 1e6:.2f}M")

    optimizer = build_optimizer(
        model,
        config["lr_backbone"],
        config["lr_head"],
        config["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["epochs"], eta_min=1e-7
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=config["label_smoothing"]
    ).to(device)

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_acc = 0.0
    logger.info("开始训练...")

    for epoch in range(config["epochs"]):
        model.train()
        clear_memory()

        total_loss, total_correct, total_count = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")

        for rgb, tac, label in pbar:
            rgb = rgb.to(device, non_blocking=True)
            tac = tac.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(rgb, tac)
                loss = criterion(logits, label)

            scaler.scale(loss).backward()

            if config["grad_clip"] > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])

            scaler.step(optimizer)
            scaler.update()

            pred = logits.argmax(dim=1)
            total_correct += (pred == label).sum().item()
            total_count += label.size(0)
            total_loss += loss.item() * label.size(0)

            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100.0 * (pred == label).float().mean().item():.2f}%"
            })

        scheduler.step()

        train_acc = 100.0 * total_correct / max(1, total_count)
        train_loss = total_loss / max(1, total_count)
        test_loss, test_acc = evaluate(model, test_loader, device, criterion)

        logger.info(
            f"[Epoch {epoch+1}/{config['epochs']}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}% | "
            f"Best: {best_acc:.2f}%"
        )

        if test_acc > best_acc:
            best_acc = test_acc

            if config["no_tgsa"]:
                ckpt_name = "best_calandra_no_tgsa.pth"
            elif config["hierarchical"]:
                ckpt_name = "best_calandra_hTrue.pth"
            else:
                ckpt_name = "best_calandra_hFalse.pth"

            save_path = os.path.join(config["save_dir"], ckpt_name)
            torch.save(model.state_dict(), save_path)
            logger.info(f"新最佳模型已保存至 {save_path}")

    logger.info(f"训练完成！最佳测试准确率: {best_acc:.2f}%")


if __name__ == "__main__":
    main()
